In [1]:
import os

In [2]:
%pwd

'c:\\Users\\KiranSreehari\\Desktop\\text summarizer\\Text-Summarizer-\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\KiranSreehari\\Desktop\\text summarizer\\Text-Summarizer-'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path
   

In [6]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [8]:
class configurationManager:
    def __init__(
            self, 
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
    
            config = self.config.model_evaluation
    
            create_directories([config.root_dir])
    
            model_evaluation_config = ModelEvaluationConfig(
                root_dir=config.root_dir,
                data_path=config.data_path,
                model_path=config.model_path,
                tokenizer_name=config.tokenizer_path,
                metric_file_name=config.metric_file_name
            )
    
            return model_evaluation_config    
            



In [11]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch
import pandas as pd
from tqdm import tqdm

In [51]:
import evaluate

class ModelEvaluation:

    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    # Evaluation
    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """Split the dataset into smaller batches that we can process simultaneously.
        Yield successive batch-sized chunks from list_of_elements.
        """

        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i:i + batch_size]

    def calculate_metric_on_test_ds(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batch_size=16,
        device="cpu",
        column_text="dialogue",
        column_summary="summary"
    ):

        article_batches = list(
            self.generate_batch_sized_chunks(
                dataset[column_text],
                batch_size
            )
        )

        target_batches = list(
            self.generate_batch_sized_chunks(
                dataset[column_summary],
                batch_size
            )
        )

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches),
            total=len(article_batches)
        ):

            inputs = tokenizer(
                article_batch,
                max_length=1024,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device),
                length_penalty=0.8,
                num_beams=8,
                max_length=128
            )

            # Decode generated summaries
            decoded_summaries = [
                tokenizer.decode(
                    s,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True
                )
                for s in summaries
            ]

            metric.add_batch(
                predictions=decoded_summaries,
                references=target_batch
            )

        # Finally compute and return the ROUGE scores
        score = metric.compute()

        return score

    def evaluate(self):

        device = "cuda" if torch.cuda.is_available() else "cpu"

        tokenizer = AutoTokenizer.from_pretrained(
            self.config.tokenizer_path
        )

        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_path
        ).to(device)

        # Loading data
        dataset_samsum_pt = load_from_disk(
            self.config.data_path
        )

        rouge_metric = evaluate.load("rouge")

        score = self.calculate_metric_on_test_ds(
            dataset_samsum_pt["test"][0:10],
            rouge_metric,
            model_pegasus,
            tokenizer,
            batch_size=2,
            device=device,
            column_text="dialogue",
            column_summary="summary"
        )

        rouge_dict = {
            "rouge1": score["rouge1"],
            "rouge2": score["rouge2"],
            "rougeL": score["rougeL"],
            "rougeLsum": score["rougeLsum"]
        }   

        df = pd.DataFrame(
            rouge_dict,
            index=["pegasus"]
        )

        df.to_csv(
            self.config.metric_file_name,
            index=False)


In [20]:
try:
    config = ConfigurationManager()

    model_evaluation_config = config.get_model_evaluation_config()

    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)

    model_evaluation_config.evaluate()

except Exception as e:
    raise e

NameError: name 'ConfigurationManager' is not defined

In [21]:
from textsummarizer.config.configuration import ConfigurationManager

In [22]:
try:
    config = ConfigurationManager()

    model_evaluation_config = config.get_model_evaluation_config()

    model_evaluation = ModelEvaluation(config=model_evaluation_config)

    model_evaluation.evaluate()

except Exception as e:
    raise e

[2026-10-06 03:02:40,360: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-06 03:02:40,369: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-06 03:02:40,373: INFO: common: created directory at: artifacts]


AttributeError: 'ConfigurationManager' object has no attribute 'get_model_evaluation_config'

In [23]:
print([x for x in dir(ConfigurationManager) if "evaluation" in x.lower()])

[]


In [26]:
import importlib
import textsummarizer.config.configuration

importlib.reload(textsummarizer.config.configuration)

from textsummarizer.config.configuration import ConfigurationManager

print("ConfigurationManager reloaded")

ImportError: cannot import name 'ModelEvaluationConfig' from 'textsummarizer.entity' (C:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-\src\textsummarizer\entity\__init__.py)

In [27]:
import inspect
from textsummarizer.config.configuration import ConfigurationManager

print(inspect.getsource(ConfigurationManager))

class ConfigurationManager:

    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:

        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )

        return data_ingestion_config

    def get_data_validation_config(self) -> DataValidationConfig:

        config = self.config.data_validation

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            r

In [28]:
config = ConfigurationManager()

print("Method exists:", hasattr(config, "get_model_evaluation_config"))

[2026-10-06 03:12:17,710: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-06 03:12:17,711: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-06 03:12:17,712: INFO: common: created directory at: artifacts]
Method exists: False


In [29]:
import sys
import importlib

if "textsummarizer.config.configuration" in sys.modules:
    del sys.modules["textsummarizer.config.configuration"]

from textsummarizer.config.configuration import ConfigurationManager

config = ConfigurationManager()

print(hasattr(config, "get_model_evaluation_config"))

ImportError: cannot import name 'ModelEvaluationConfig' from 'textsummarizer.entity' (C:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-\src\textsummarizer\entity\__init__.py)

In [30]:
from textsummarizer.entity import ModelEvaluationConfig

print("ModelEvaluationConfig imported successfully")

ImportError: cannot import name 'ModelEvaluationConfig' from 'textsummarizer.entity' (C:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-\src\textsummarizer\entity\__init__.py)

In [31]:
from pathlib import Path

entity_file = Path("../src/textsummarizer/entity/__init__.py")

print(entity_file.resolve())
print("ModelEvaluationConfig" in entity_file.read_text(encoding="utf-8"))

C:\Users\KiranSreehari\Desktop\text summarizer\src\textsummarizer\entity\__init__.py


FileNotFoundError: [Errno 2] No such file or directory: '..\\src\\textsummarizer\\entity\\__init__.py'

In [32]:
import os
print(os.getcwd())

c:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-


In [33]:
from pathlib import Path

entity_file = Path("src/textsummarizer/entity/__init__.py")

print(entity_file.resolve())
print("ModelEvaluationConfig" in entity_file.read_text(encoding="utf-8"))

C:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-\src\textsummarizer\entity\__init__.py
True


In [34]:
import importlib
import textsummarizer.entity

importlib.reload(textsummarizer.entity)

from textsummarizer.entity import ModelEvaluationConfig

print("ModelEvaluationConfig imported successfully")

ModelEvaluationConfig imported successfully


In [35]:
import importlib
import textsummarizer.config.configuration

importlib.reload(textsummarizer.config.configuration)

from textsummarizer.config.configuration import ConfigurationManager

config = ConfigurationManager()

print(hasattr(config, "get_model_evaluation_config"))

[2026-10-06 03:16:22,682: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-06 03:16:22,685: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-06 03:16:22,686: INFO: common: created directory at: artifacts]
True


In [36]:
model_evaluation_config = config.get_model_evaluation_config()

print(model_evaluation_config)

[2026-10-06 03:16:55,930: INFO: common: created directory at: artifacts/model_evaluation]
ModelEvaluationConfig(root_dir=WindowsPath('artifacts/model_evaluation'), data_path=WindowsPath('artifacts/data_transformation/samsum_dataset'), model_path=WindowsPath('artifacts/model_trainer/pegasus-samsum-model'), tokenizer_path=WindowsPath('artifacts/model_trainer/tokenizer'), metric_file_name=WindowsPath('artifacts/model_evaluation/metrics.csv'))


In [37]:
model_evaluation = ModelEvaluation(config=model_evaluation_config)

print("ModelEvaluation object created successfully")

ModelEvaluation object created successfully


In [38]:
print("Ready to run evaluation")
print("Model path:", model_evaluation_config.model_path)
print("Tokenizer path:", model_evaluation_config.tokenizer_path)
print("Data path:", model_evaluation_config.data_path)


Ready to run evaluation
Model path: artifacts\model_trainer\pegasus-samsum-model
Tokenizer path: artifacts\model_trainer\tokenizer
Data path: artifacts\data_transformation\samsum_dataset


In [39]:
try:
    model_evaluation.evaluate()
    print("Evaluation completed successfully")

except Exception as e:
    raise e

OSError: Incorrect path_or_model_id: 'artifacts\model_trainer\tokenizer'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [40]:
import importlib
import textsummarizer.config.configuration

importlib.reload(textsummarizer.config.configuration)

from textsummarizer.config.configuration import ConfigurationManager

config = ConfigurationManager()

model_evaluation_config = config.get_model_evaluation_config()

print("Tokenizer path:", model_evaluation_config.tokenizer_path)

[2026-10-06 03:19:52,291: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-06 03:19:52,297: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-06 03:19:52,301: INFO: common: created directory at: artifacts]
[2026-10-06 03:19:52,301: INFO: common: created directory at: artifacts/model_evaluation]
Tokenizer path: tokenizer


In [41]:
model_evaluation = ModelEvaluation(config=model_evaluation_config)

print("ModelEvaluation updated successfully")

ModelEvaluation updated successfully


In [42]:
try:
    model_evaluation.evaluate()
    print("Evaluation completed successfully")

except Exception as e:
    raise e

NameError: name 'load_metric' is not defined

In [43]:
import inspect

print(inspect.getfile(ModelEvaluation))

OSError: source code not available

In [44]:
import inspect

print(inspect.getsource(ModelEvaluation))

OSError: source code not available

In [45]:
from textsummarizer.components.model_evaluation import ModelEvaluation

ModuleNotFoundError: No module named 'textsummarizer.components.model_evaluation'

In [52]:
try:
    config = ConfigurationManager()

    model_evaluation_config = config.get_model_evaluation_config()

    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)

    model_evaluation_config.evaluate()

except Exception as e:
    raise e

[2026-10-06 03:38:33,299: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-06 03:38:33,302: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-06 03:38:33,305: INFO: common: created directory at: artifacts]
[2026-10-06 03:38:33,307: INFO: common: created directory at: artifacts/model_evaluation]


100%|██████████| 5/5 [04:07<00:00, 49.58s/it]

[2026-10-06 03:42:52,603: INFO: rouge_scorer: Using default tokenizer.]
